# Chapter 15 · The Meaning of a P-Value: Intervals, Noninferiority, and Interim Analyses: companion notebook
*Data Science and Artificial Intelligence in Health, Volume 1*

> **This notebook accompanies the book chapter; it does not reproduce it.** Definitions, derivations, and explanations are in the book. Here you run the calculations on the case and check the numbers. All data are **synthetic**, and the values are for teaching purposes only.

**Part A** follows the chapter's case, a randomized trial of 7 versus 14 days of antibiotics and the stewardship committee that reads it, through the committee's ten questions. **Part B** gives the written solution of every chapter exercise (statements are in the book) and, for the exercises whose solution needs a calculation, its code and result. Run the cells in order. The packages are listed in `requirements.txt`, and the notebook reads `data/trial.csv` from its own folder.

## Part A. The case

**Data.** `data/trial.csv`: 1,200 rows, one per enrolled patient, and six columns: `patient_id`, `site` (S01 to S12), `month` (1 to 24), `allocated` (short or standard), `received` (short or standard), `failure_30d` (yes or no). Each cell carries the number of the book section it belongs to.

### 15.1 Introduction: the data

In [1]:
# 1. Read the trial file: one row is one enrolled patient
import numpy as np
import pandas as pd
from math import sqrt, ceil, log2
from scipy import stats
from scipy.stats import norm, multivariate_normal
from scipy.optimize import brentq
from scipy.integrate import quad

df = pd.read_csv("data/trial.csv")

# 2. Count rows and columns
rows, columns = df.shape
print("The trial file has", rows, "rows (one per patient) and", columns, "columns.")

# 3. Add a helper column that is 1 for a failure and 0 otherwise (the notebook's calculations use it)
df["fail"] = (df["failure_30d"] == "yes").astype(int)
df.head()                                      # The first five rows

The trial file has 1200 rows (one per patient) and 6 columns.


,patient_id,site,month,allocated,received,failure_30d,fail
0,P0001,S01,1,standard,standard,no,0
1,P0002,S02,1,standard,standard,no,0
2,P0003,S03,1,short,short,no,0
3,P0004,S04,1,standard,standard,no,0
4,P0005,S05,1,short,short,no,0


### 15.2 Noninferiority: the margin and the test

*Is the 7-day course not worse than the 14-day course by more than 5 points?*

In [2]:
# 1. Failures in each arm, by allocation
short = df[df["allocated"] == "short"]["fail"]
standard = df[df["allocated"] == "standard"]["fail"]
n_s, k_s, p_s = len(short), int(short.sum()), short.mean()
n_c, k_c, p_c = len(standard), int(standard.sum()), standard.mean()
print("Short:", k_s, "of", n_s, "(", round(100 * p_s, 1), "%); standard:", k_c, "of", n_c, "(", round(100 * p_c, 1), "%)")

# 2. Difference, standard error, upper limit, and the test against the margin of 0.05
margin = 0.05
theta = p_s - p_c
se = sqrt(p_s * (1 - p_s) / n_s + p_c * (1 - p_c) / n_c)
z_ni = (theta - margin) / se
print("Difference:", round(100 * theta, 2), "points; standard error:", "%.2f" % (100 * se), "points")
print("95% CI:", round(100 * (theta - 1.96 * se), 1), "to", round(100 * (theta + 1.96 * se), 1), "; z =", round(z_ni, 2), "; one-sided p =", round(norm.cdf(z_ni), 4))

# 3. Reading: noninferiority is shown when the upper limit is below the margin
if theta + 1.96 * se < margin:
    print("The upper limit is below the margin of 5 points: noninferiority is shown by allocation.")
else:
    print("The upper limit is not below the margin of 5 points: noninferiority is not shown by allocation.")

Short: 68 of 591 ( 11.5 %); standard: 63 of 609 ( 10.3 %)
Difference: 1.16 points; standard error: 1.80 points
95% CI: -2.4 to 4.7 ; z = -2.13 ; one-sided p = 0.0166
The upper limit is below the margin of 5 points: noninferiority is shown by allocation.


### 15.3 Intention to treat and per protocol

*Does the answer hold for the patients who received the course they were assigned?*

In [3]:
# 1. Patients who did not receive the course they were assigned
deviated = df[df["allocated"] != df["received"]]
print("Deviated:", len(deviated), "(short to standard:", int(((df["allocated"] == "short") & (df["received"] == "standard")).sum()), "; standard to short:", int(((df["allocated"] == "standard") & (df["received"] == "short")).sum()), ")")
print("Failure among deviators:", round(100 * deviated["fail"].mean(), 1), "%; among the others:", round(100 * df[df["allocated"] == df["received"]]["fail"].mean(), 1), "%")

# 2. Per-protocol analysis: patients who received what they were assigned
pp = df[df["allocated"] == df["received"]]
pp_s = pp[pp["allocated"] == "short"]["fail"]
pp_c = pp[pp["allocated"] == "standard"]["fail"]
theta_pp = pp_s.mean() - pp_c.mean()
se_pp = sqrt(pp_s.mean() * (1 - pp_s.mean()) / len(pp_s) + pp_c.mean() * (1 - pp_c.mean()) / len(pp_c))
z_pp = (theta_pp - margin) / se_pp
print("Per protocol:", len(pp), "patients;", int(pp_s.sum()), "of", len(pp_s), "and", int(pp_c.sum()), "of", len(pp_c))
print("Difference:", round(100 * theta_pp, 2), "points; standard error:", "%.2f" % (100 * se_pp), "; upper limit:", round(100 * (theta_pp + 1.96 * se_pp), 2), "; z =", round(z_pp, 2), "; one-sided p =", round(norm.cdf(z_pp), 3))

# 3. Reading
if theta_pp + 1.96 * se_pp < margin:
    print("Per protocol the upper limit is below the margin: noninferiority is shown.")
else:
    print("Per protocol the upper limit is not below the margin: noninferiority is not shown.")

Deviated: 68 (short to standard: 43 ; standard to short: 25 )
Failure among deviators: 16.2 %; among the others: 10.6 %
Per protocol: 1132 patients; 63 of 548 and 57 of 584
Difference: 1.74 points; standard error: 1.83 ; upper limit: 5.33 ; z = -1.78 ; one-sided p = 0.038
Per protocol the upper limit is not below the margin: noninferiority is not shown.


### 15.4 Equivalence and two one-sided tests

*Would the answer change if the aim were to show that the two courses are equivalent?*

In [4]:
# 1. Two one-sided tests against the margins -0.05 and +0.05
z_low = (theta + margin) / se
p_low = norm.sf(z_low)
p_up = norm.cdf((theta - margin) / se)
print("Lower test: z =", round(z_low, 2), "; p =", round(p_low, 5), "; upper test: p =", round(p_up, 4))

# 2. The TOST p-value is the larger of the two
print("TOST p =", round(max(p_low, p_up), 4), "; 95% CI inside the margins:", theta - 1.96 * se > -margin and theta + 1.96 * se < margin)
if max(p_low, p_up) < 0.025:
    print("Both one-sided tests reject: equivalence within 5 points is shown by allocation.")

Lower test: z = 3.42 ; p = 0.00031 ; upper test: p = 0.0166
TOST p = 0.0166 ; 95% CI inside the margins: True
Both one-sided tests reject: equivalence within 5 points is shown by allocation.


### 15.5 Sample size and power of a noninferiority trial

*How many patients did the trial need?*

In [5]:
# 1. Patients per arm for a noninferiority test with equal courses and a standard failure rate of 9.5%
za = norm.ppf(0.975)
p0 = 0.095
for power, zb in (("80%", norm.ppf(0.80)), ("90%", norm.ppf(0.90))):
    n_arm = (za + zb) ** 2 * (2 * p0 * (1 - p0)) / margin ** 2
    print("Power", power, ": patients per arm", round(n_arm, 1), "-> rounded up", ceil(n_arm))

# 2. Power of 600 patients per arm for a true loss of 0, 1, 2, and 3 points
for loss in (0.0, 0.01, 0.02, 0.03):
    se_true = sqrt(p0 * (1 - p0) / 600 + (p0 + loss) * (1 - p0 - loss) / 600)
    print("True loss (points):", round(100 * loss), "; power", round(norm.cdf((margin - loss) / se_true - za), 3))

# 3. Patients per arm for 80% power when the short course is truly 2 points worse
loss = 0.02
print("Patients per arm for a 2-point loss:", ceil((za + 0.8416) ** 2 * (p0 * (1 - p0) + (p0 + loss) * (1 - p0 - loss)) / (margin - loss) ** 2))

# 4. Simulation: 10,000 trials of 600 per arm; share declaring noninferiority for several true losses
rng = np.random.default_rng(15)
for loss in (0.05, 0.0, 0.01, 0.02):
    standard_rate = rng.binomial(600, p0, 10000) / 600
    short_rate = rng.binomial(600, p0 + loss, 10000) / 600
    standard_vs_short = short_rate - standard_rate
    se_sim = np.sqrt(standard_rate * (1 - standard_rate) / 600 + short_rate * (1 - short_rate) / 600)
    print("True loss (points):", round(100 * loss), "; noninferiority declared in", round(100 * ((standard_vs_short) + 1.96 * se_sim < margin).mean(), 1), "% of the trials")

Power 80% : patients per arm 539.8 -> rounded up 540
Power 90% : patients per arm 722.7 -> rounded up 723
True loss (points): 0 ; power 0.84
True loss (points): 1 ; power 0.637
True loss (points): 2 ; power 0.396
True loss (points): 3 ; power 0.197
Patients per arm for a 2-point loss: 1638
True loss (points): 5 ; noninferiority declared in 2.8 % of the trials
True loss (points): 0 ; noninferiority declared in 83.8 % of the trials
True loss (points): 1 ; noninferiority declared in 63.8 % of the trials
True loss (points): 2 ; noninferiority declared in 40.6 % of the trials


### 15.6 Confidence intervals and tests are one calculation

*Do the confidence interval and the test say the same thing?*

In [6]:
# 1. Tests of two hypothesized differences and the limits of the 95% interval
for theta0 in (0.0, 0.05, theta - 1.96 * se, theta + 1.96 * se):
    print("Hypothesized difference", round(100 * theta0, 2), "points: two-sided p =", round(2 * norm.sf(abs((theta - theta0) / se)), 4))

# 2. The 95% interval and whether it contains 0 and 0.05
low = theta - 1.96 * se
high = theta + 1.96 * se
print("95% CI:", round(100 * low, 2), "to", round(100 * high, 2), "; contains 0:", low < 0 < high, "; contains 0.05:", low < 0.05 < high)

Hypothesized difference 0.0 points: two-sided p = 0.5193
Hypothesized difference 5.0 points: two-sided p = 0.0331
Hypothesized difference -2.37 points: two-sided p = 0.05
Hypothesized difference 4.69 points: two-sided p = 0.05
95% CI: -2.37 to 4.69 ; contains 0: True ; contains 0.05: False


### 15.7 The p-value function

*Which differences between the courses are compatible with the data?*

In [7]:
# 1. P-value function at several hypothesized differences (points)
for theta0 in (-4, 0, 100 * theta, 2, 3, 4, 5, 6, 7):
    print("Hypothesized", round(theta0, 2), "points: p =", round(2 * norm.sf(abs((theta - theta0 / 100) / se)), 4))

# 2. The two hypotheses with the same p-value as zero, and the 90% interval
print("Same p-value as 0:", 0, "and", round(200 * theta, 2), "points; 90% CI:", round(100 * (theta - 1.645 * se), 2), "to", round(100 * (theta + 1.645 * se), 2))

Hypothesized -4 points: p = 0.0042
Hypothesized 0 points: p = 0.5193
Hypothesized 1.16 points: p = 1.0
Hypothesized 2 points: p = 0.6415
Hypothesized 3 points: p = 0.3074
Hypothesized 4 points: p = 0.1151
Hypothesized 5 points: p = 0.0331
Hypothesized 6 points: p = 0.0072
Hypothesized 7 points: p = 0.0012
Same p-value as 0: 0 and 2.32 points; 90% CI: -1.8 to 4.12


### 15.8 What a p-value means

*What does a p-value of 0.52 for no difference tell the committee, and what does it not?*

In [8]:
# 1. S-values: bits of surprise, -log2(p)
for p in (0.52, 0.05, 0.033, 0.001):
    print("p =", p, ": S-value", round(-log2(p), 2), "bits")

# 2. The p-values of this trial in bits
print("Test of no difference:", round(-log2(2 * norm.sf(abs(theta / se))), 2), "bits; test of a 5-point loss:", round(-log2(2 * norm.sf(abs((theta - margin) / se))), 2), "bits")

p = 0.52 : S-value 0.94 bits
p = 0.05 : S-value 4.32 bits
p = 0.033 : S-value 4.92 bits
p = 0.001 : S-value 9.97 bits
Test of no difference: 0.95 bits; test of a 5-point loss: 4.92 bits


### 15.9 What a p-value does not mean

*What would the committee be wrong to conclude from the p-values of this trial?*

In [9]:
# 1. Resample the trial's patients 2,000 times and recompute the one-sided noninferiority p-value
rng = np.random.default_rng(15)
y = df["fail"].values
idx_s = np.where(df["allocated"] == "short")[0]
idx_c = np.where(df["allocated"] == "standard")[0]
p_boot = []
for repetition in range(2000):
    xs = y[rng.choice(idx_s, len(idx_s))]
    xc = y[rng.choice(idx_c, len(idx_c))]
    se_b = sqrt(xs.mean() * (1 - xs.mean()) / len(xs) + xc.mean() * (1 - xc.mean()) / len(xc))
    p_boot.append(norm.cdf((xs.mean() - xc.mean() - margin) / se_b))
p_boot = np.array(p_boot)
print("Smallest p:", float("%.2g" % p_boot.min()), "; median:", round(np.median(p_boot), 4), "; largest:", round(p_boot.max(), 3), "; rule met in", round(100 * (p_boot < 0.025).mean(), 1), "% of the resamples")

# 2. Power computed at the observed estimate, for the test of no difference
z_obs = theta / se
print("Observed z:", round(z_obs, 2), "; two-sided p =", round(2 * norm.sf(abs(z_obs)), 3), "; observed power:", round(norm.sf(1.96 - z_obs) + norm.cdf(-1.96 - z_obs), 3))

Smallest p: 3.4e-08 ; median: 0.0167 ; largest: 0.77 ; rule met in 55.9 % of the resamples
Observed z: 0.64 ; two-sided p = 0.519 ; observed power: 0.099


### 15.10 Significant versus important

*If a larger trial had made the difference of 1.2 points significant, would that change what the committee should do?*

In [10]:
# 1. The test of no difference with the observed difference and failure rates held fixed, for larger trials
for n_arm in (591, 3000, 10000, 30000):
    se_n = sqrt((p_s * (1 - p_s) + p_c * (1 - p_c)) / n_arm)       # standard error with n_arm patients in each arm
    z_n = theta / se_n
    print("Patients per arm", n_arm, ": z =", round(z_n, 2), "; two-sided p =", "%.2g" % (2 * norm.sf(abs(z_n))))

Patients per arm 591 : z = 0.64 ; two-sided p = 0.52
Patients per arm 3000 : z = 1.44 ; two-sided p = 0.15
Patients per arm 10000 : z = 2.63 ; two-sided p = 0.0085
Patients per arm 30000 : z = 4.56 ; two-sided p = 5.1e-06


### 15.11 How many significant results are false

*How many of the significant results of such trials are false?*

In [11]:
# 1. Share of significant results that are false, for each share of real hypotheses and power (alpha = 0.05)
for real in (0.5, 0.1, 0.01):
    shares = []
    for power in (0.8, 0.5, 0.2):
        shares.append(round((1 - real) * 0.05 / ((1 - real) * 0.05 + real * power), 3))
    print("Real hypotheses", round(100 * real), "%: power 0.8, 0.5, 0.2 ->", shares)

# 2. The example of 1,000 hypotheses, 100 of them real, power 0.50
false_positives = 900 * 0.05
true_positives = 100 * 0.50
print("False positives:", false_positives, "; true positives:", true_positives, "; share false:", round(false_positives / (false_positives + true_positives), 3))

Real hypotheses 50 %: power 0.8, 0.5, 0.2 -> [0.059, 0.091, 0.2]
Real hypotheses 10 %: power 0.8, 0.5, 0.2 -> [0.36, 0.474, 0.692]
Real hypotheses 1 %: power 0.8, 0.5, 0.2 -> [0.861, 0.908, 0.961]
False positives: 45.0 ; true positives: 50.0 ; share false: 0.474


### 15.12 Many tests: multiplicity and the Bonferroni correction

*One site shows a benefit of the short course: is it real?*

In [12]:
# 1. Difference and Fisher's exact p-value at each of the 12 sites
site_p = []
for site, group in df.groupby("site"):
    a = group[group["allocated"] == "short"]["fail"]
    b = group[group["allocated"] == "standard"]["fail"]
    table = [[int(a.sum()), len(a) - int(a.sum())], [int(b.sum()), len(b) - int(b.sum())]]
    p_site = stats.fisher_exact(table)[1]
    site_p.append(p_site)
    print(site, ": short", int(a.sum()), "of", len(a), "; standard", int(b.sum()), "of", len(b), "; Fisher p =", round(p_site, 4))    # p = 1.0: almost the same failure rate in both arms
print("Sites with p < 0.05:", sum(p < 0.05 for p in site_p), "; Bonferroni threshold:", round(0.05 / 12, 4), "; chance of at least one in 12 independent tests:", round(1 - 0.95 ** 12, 3))

# 2. Shuffle the allocation labels 2,000 times (no difference by construction) and count the smallest site p-value
rng = np.random.default_rng(15)
y = df["fail"].values
labels = (df["allocated"] == "short").values
site_ix = [np.where(df["site"].values == s)[0] for s in sorted(df["site"].unique())]
any_005 = 0
any_bonf = 0
for repetition in range(2000):
    shuffled = rng.permutation(labels)
    smallest = 1
    for ix in site_ix:
        is_short = shuffled[ix]
        yy = y[ix]
        table = [[yy[is_short].sum(), is_short.sum() - yy[is_short].sum()], [yy[~is_short].sum(), (~is_short).sum() - yy[~is_short].sum()]]
        smallest = min(smallest, stats.fisher_exact(table)[1])
    any_005 = any_005 + (smallest < 0.05)
    any_bonf = any_bonf + (smallest < 0.05 / 12)
print("Shuffled trials with a site p < 0.05:", round(100 * any_005 / 2000, 1), "%; with a site p < 0.0042:", round(100 * any_bonf / 2000, 1), "%")

S01 : short 8 of 46 ; standard 3 of 54 ; Fisher p = 0.1058
S02 : short 7 of 52 ; standard 3 of 48 ; Fisher p = 0.3222
S03 : short 7 of 56 ; standard 4 of 44 ; Fisher p = 0.751
S04 : short 6 of 43 ; standard 6 of 57 ; Fisher p = 0.7577
S05 : short 3 of 46 ; standard 3 of 54 ; Fisher p = 1.0
S06 : short 4 of 44 ; standard 6 of 56 ; Fisher p = 1.0
S07 : short 5 of 34 ; standard 9 of 66 ; Fisher p = 1.0
S08 : short 4 of 56 ; standard 5 of 44 ; Fisher p = 0.5013
S09 : short 0 of 53 ; standard 5 of 47 ; Fisher p = 0.0204
S10 : short 9 of 48 ; standard 6 of 52 ; Fisher p = 0.4038
S11 : short 11 of 60 ; standard 7 of 40 ; Fisher p = 1.0
S12 : short 4 of 53 ; standard 6 of 47 ; Fisher p = 0.5092
Sites with p < 0.05: 1 ; Bonferroni threshold: 0.0042 ; chance of at least one in 12 independent tests: 0.46


Shuffled trials with a site p < 0.05: 33.3 %; with a site p < 0.0042: 2.8 %


### 15.13 Looking at the data again and again

*Could the committee have looked at the data before the trial ended, and stopped?*

In [13]:
# 1. Shuffle the allocation labels 4,000 times and test after every 200 patients (six looks), two-sided
rng = np.random.default_rng(15)
labels = (df["allocated"] == "short").values
y = df["fail"].values
looks = [200, 400, 600, 800, 1000, 1200]
reject_196 = 0
reject_pocock = 0
reject_final = 0
for repetition in range(4000):
    shuffled = rng.permutation(labels)
    z_looks = []
    for n_look in looks:
        is_short = shuffled[:n_look]
        yy = y[:n_look]
        p_all = yy.mean()                                          # failure rate of all patients so far: the standard error under the null hypothesis
        se_look = sqrt(p_all * (1 - p_all) * (1 / is_short.sum() + 1 / (n_look - is_short.sum())))
        z_looks.append(abs(yy[is_short].mean() - yy[~is_short].mean()) / se_look)
    z_looks = np.array(z_looks)
    reject_196 = reject_196 + (z_looks > 1.96).any()
    reject_pocock = reject_pocock + (z_looks > 2.453).any()
    reject_final = reject_final + (z_looks[-1] > 1.96)
print("True null rejected: final look only", round(100 * reject_final / 4000, 1), "%; six looks at 1.96:", round(100 * reject_196 / 4000, 1), "%; six looks at 2.453:", round(100 * reject_pocock / 4000, 1), "%")
print("If the six looks were independent:", round(100 * (1 - 0.95 ** 6), 1), "%")

True null rejected: final look only 4.4 %; six looks at 1.96: 16.0 %; six looks at 2.453: 5.1 %
If the six looks were independent: 26.5 %


### 15.14 Group-sequential boundaries and alpha spending

*How should the committee have set its boundaries for three looks, at 400, 800, and 1,200 patients?*

In [14]:
# 1. Correlation of the statistics at equally spaced looks: sqrt(t_i / t_j)
n_looks = 3
t = np.arange(1, n_looks + 1) / n_looks                       # information fractions 1/3, 2/3, 1
corr = np.sqrt(np.minimum.outer(t, t) / np.maximum.outer(t, t))
print("Correlation matrix of three looks:", np.round(corr, 3).tolist())

# 2. Pocock and O'Brien-Fleming boundaries for three looks, total one-sided level 0.025 (two-sided 0.05)
# 1 minus the joint normal probability of no crossing is the probability of crossing at any look; brentq searches for the boundary that makes it 0.025
mv3 = multivariate_normal(mean=np.zeros(3), cov=corr, seed=1, abseps=1e-8, releps=1e-8)
pocock = brentq(lambda c: 1 - mv3.cdf(np.array([c, c, c])) - 0.025, 1.5, 4)
obf_c = brentq(lambda c: 1 - mv3.cdf(np.array([c * sqrt(3 / k) for k in (1, 2, 3)])) - 0.025, 1.5, 4)
print("Pocock:", round(pocock, 3), "; O'Brien-Fleming:", [round(obf_c * sqrt(3 / k), 3) for k in (1, 2, 3)])
print("Nominal two-sided levels: Pocock", round(2 * norm.sf(pocock), 4), "; O'Brien-Fleming", [round(float(2 * norm.sf(obf_c * sqrt(3 / k))), 4) for k in (1, 2, 3)])

# 3. Lan-DeMets spending function at one third, two thirds, and the end
print("Spending:", [round(float(2 - 2 * norm.cdf(1.96 / sqrt(t_k))), 4) for t_k in (1 / 3, 2 / 3, 1)])

# 4. The Pocock boundary for six looks
t6 = np.arange(1, 7) / 6
mv6 = multivariate_normal(mean=np.zeros(6), cov=np.sqrt(np.minimum.outer(t6, t6) / np.maximum.outer(t6, t6)), seed=1, abseps=1e-8, releps=1e-8)
print("Pocock, six looks:", round(brentq(lambda c: 1 - mv6.cdf(np.array([c] * 6)) - 0.025, 1.5, 4), 3))

# 5. The trial at 400, 800, and 1,200 patients: evidence for noninferiority against the O'Brien-Fleming boundaries
evidence = []
for n_look, bound in ((400, obf_c * sqrt(3)), (800, obf_c * sqrt(3 / 2)), (1200, obf_c)):
    part = df.iloc[:n_look]
    a = part[part["allocated"] == "short"]["fail"]
    b = part[part["allocated"] == "standard"]["fail"]
    se_look = sqrt(a.mean() * (1 - a.mean()) / len(a) + b.mean() * (1 - b.mean()) / len(b))
    z_look = -(a.mean() - b.mean() - margin) / se_look
    evidence.append(z_look)
    print("Look at", n_look, ": difference", round(100 * (a.mean() - b.mean()), 2), "points; evidence", round(z_look, 2), "; boundary", round(bound, 2), "; crossed:", z_look > bound)
print("Final upper limit with the boundary:", round(100 * (theta + obf_c * se), 2), "points")

Correlation matrix of three looks: [[1.0, 0.707, 0.577], [0.707, 1.0, 0.816], [0.577, 0.816, 1.0]]


Pocock: 2.289 ; O'Brien-Fleming: [3.471, 2.454, 2.004]
Nominal two-sided levels: Pocock 0.0221 ; O'Brien-Fleming [0.0005, 0.0141, 0.0451]
Spending: [0.0007, 0.0164, 0.05]


Pocock, six looks: 2.453
Look at 400 : difference 2.04 points; evidence 1.02 ; boundary 3.47 ; crossed: False
Look at 800 : difference 1.32 points; evidence 1.68 ; boundary 2.45 ; crossed: False
Look at 1200 : difference 1.16 points; evidence 2.13 ; boundary 2.0 ; crossed: True
Final upper limit with the boundary: 4.77 points


### 15.15 Stopping early: benefit, futility, harm, and the monitoring committee

*For what reasons would the trial have been stopped, and who decides?*

In [15]:
# 1. Conditional power at the second look (information fraction 2/3), with the evidence of the second look and the final boundary
t_look = 2 / 3
z_look = evidence[1]
c_final = obf_c
print("Evidence at the second look:", round(z_look, 2), "; final boundary:", round(c_final, 3))
print("Conditional power:", round(norm.cdf((z_look / sqrt(t_look) - c_final) / sqrt(1 - t_look)), 3))

Evidence at the second look: 1.68 ; final boundary: 2.004
Conditional power: 0.535


### 15.16 The estimate after early stopping

*If the trial had stopped early, would its estimate be reliable?*

In [16]:
# 1. 10,000 simulated trials, 200 patients per arm per look, O'Brien-Fleming boundaries; true loss 2 points
rng = np.random.default_rng(15)
bounds = [obf_c * sqrt(3 / k) for k in (1, 2, 3)]                 # the O'Brien-Fleming boundaries of Section 15.14
p_standard = 0.095
true_loss = 0.02
stopped = {1: [], 2: [], 3: []}
for repetition in range(10000):
    standard_arm = rng.random(600) < p_standard
    short_arm = rng.random(600) < p_standard + true_loss
    for look in (1, 2, 3):
        n_look = 200 * look
        rate_standard = standard_arm[:n_look].mean()
        rate_short = short_arm[:n_look].mean()
        se_look = sqrt(rate_standard * (1 - rate_standard) / n_look + rate_short * (1 - rate_short) / n_look)
        if se_look > 0 and (rate_short - rate_standard - margin) / se_look < -bounds[look - 1]:
            stopped[look].append(rate_short - rate_standard)
            break

# 2. Where the trials declared noninferiority and what they estimated
declared = stopped[1] + stopped[2] + stopped[3]
print("Declared noninferiority:", round(100 * len(declared) / 10000, 1), "% of the trials; by look:", len(stopped[1]), len(stopped[2]), len(stopped[3]))
print("Mean estimate at each look (points):", [round(float(100 * np.mean(stopped[k])), 1) for k in (1, 2, 3)], "; among all declared:", round(float(100 * np.mean(declared)), 1), "; true: 2.0")

Declared noninferiority: 40.2 % of the trials; by look: 91 1417 2514
Mean estimate at each look (points): [-6.3, -1.3, 0.6] ; among all declared: -0.2 ; true: 2.0


## Part B. Exercise solutions

Part B reuses variables created in Part A (`theta`, `se`, `margin`, `theta_pp`, `se_pp`, `obf_c`): run Part A first.

### 15.18 Exercise 1. Margins of 3 and 6 points

By allocation, the difference is 1.16 points with a standard error of 1.80, so the upper limit of the 95% interval is 4.69 points in both cases. With a margin of 3 points, $z=(1.16-3)/1.80=-1.02$, the one-sided $p=0.154$, and the upper limit is above the margin: noninferiority is not shown. With a margin of 6 points, $z=-2.69$, $p=0.0036$, and the upper limit is below the margin: noninferiority is shown. The conclusion depends on the margin, which is why it is fixed before the trial.

In [17]:
# 1. Noninferiority by allocation with margins of 3 and 6 points
for m in (0.03, 0.06):
    z_m = (theta - m) / se
    print("Margin (points):", round(100 * m), "; upper limit", round(100 * (theta + 1.96 * se), 2), "; z =", round(z_m, 2), "; one-sided p =", round(norm.cdf(z_m), 4), "; noninferior:", theta + 1.96 * se < m)

Margin (points): 3 ; upper limit 4.69 ; z = -1.02 ; one-sided p = 0.1537 ; noninferior: False
Margin (points): 6 ; upper limit 4.69 ; z = -2.69 ; one-sided p = 0.0036 ; noninferior: True


### 15.18 Exercise 2. Per protocol and as treated

Per protocol (1,132 patients), the difference is 1.74 points and the upper limit is 5.33, above the margin: not shown. As treated (573 patients who received the short course and 627 who received the standard course), the difference is 2.15 points, the upper limit is 5.70, and $z=-1.57$: not shown. The as-treated analysis is not randomized, because the patients who deviated are not a random subset: their failure rate was 16.2%, against 10.6% in the others, so grouping by the course received mixes the effect of the course with the reasons for deviating.

In [18]:
# 1. Per protocol and as treated (by the course received, all 1,200 patients)
rec_s = df[df["received"] == "short"]["fail"]
rec_c = df[df["received"] == "standard"]["fail"]
theta_at = rec_s.mean() - rec_c.mean()
se_at = sqrt(rec_s.mean() * (1 - rec_s.mean()) / len(rec_s) + rec_c.mean() * (1 - rec_c.mean()) / len(rec_c))
print("Per protocol: difference", round(100 * theta_pp, 2), "; upper limit", round(100 * (theta_pp + 1.96 * se_pp), 2))
print("As treated: short", int(rec_s.sum()), "of", len(rec_s), "; standard", int(rec_c.sum()), "of", len(rec_c))
print("Difference", round(100 * theta_at, 2), "; standard error", "%.2f" % (100 * se_at), "; upper limit", round(100 * (theta_at + 1.96 * se_at), 2), "; z =", round((theta_at - margin) / se_at, 2), "; one-sided p =", round(norm.cdf((theta_at - margin) / se_at), 3))

Per protocol: difference 1.74 ; upper limit 5.33
As treated: short 69 of 573 ; standard 62 of 627
Difference 2.15 ; standard error 1.81 ; upper limit 5.7 ; z = -1.57 ; one-sided p = 0.058


### 15.18 Exercise 3. Sample sizes and power

For equal courses, $p_c=0.095$, $\Delta=0.05$, the formula gives 540 patients per arm for 80% power (539.9 before rounding up) and 723 for 90% (722.7). If the short course is truly 1 point worse, 80% power needs 883 per arm. The power of 600 per arm is 0.84 for a true loss of 0 points, 0.64 for 1 point, 0.40 for 2 points, and 0.20 for 3 points. The closer the true loss is to the margin, the larger the trial must be, because the denominator $(\Delta-\theta)^{2}$ shrinks.

In [19]:
# 1. Sample sizes and power of 600 per arm
print("Equal courses: 80%", ceil((1.96 + 0.8416) ** 2 * (2 * 0.095 * 0.905) / 0.05 ** 2), "; 90%", ceil((1.96 + 1.2816) ** 2 * (2 * 0.095 * 0.905) / 0.05 ** 2))
print("True loss 1 point, 80%:", ceil((1.96 + 0.8416) ** 2 * (0.095 * 0.905 + 0.105 * 0.895) / 0.04 ** 2))
for loss in (0.0, 0.01, 0.02, 0.03):
    se_true = sqrt(0.095 * 0.905 / 600 + (0.095 + loss) * (0.905 - loss) / 600)
    print("True loss (points):", round(100 * loss), "; power", round(norm.cdf((0.05 - loss) / se_true - 1.96), 3))

Equal courses: 80% 540 ; 90% 723
True loss 1 point, 80%: 883
True loss (points): 0 ; power 0.84
True loss (points): 1 ; power 0.637
True loss (points): 2 ; power 0.396
True loss (points): 3 ; power 0.197


### 15.18 Exercise 4. Equivalence tests

With margins of 5 points, the lower test has $p=0.0003$ and the upper test $p=0.0166$; the TOST p-value is the larger, 0.017, below 0.025: equivalence is shown. With margins of 3 points, the lower test has $p=0.010$ and the upper $p=0.154$; the TOST p-value is 0.154: equivalence within 3 points is not shown. The 95% interval ($-2.4$ to 4.7) lies inside $-5$ to 5, and not inside $-3$ to 3.

In [20]:
# 1. Equivalence tests with margins of 5 and 3 points
for m in (0.05, 0.03):
    p_lo = norm.sf((theta + m) / se)
    p_hi = norm.cdf((theta - m) / se)
    print("Margin (points):", round(100 * m), "; lower p =", round(p_lo, 5), "; upper p =", round(p_hi, 4), "; TOST p =", round(max(p_lo, p_hi), 4), "; equivalent at 0.025:", max(p_lo, p_hi) < 0.025)

Margin (points): 5 ; lower p = 0.00031 ; upper p = 0.0166 ; TOST p = 0.0166 ; equivalent at 0.025: True
Margin (points): 3 ; lower p = 0.01045 ; upper p = 0.1537 ; TOST p = 0.1537 ; equivalent at 0.025: False


### 15.18 Exercise 5. The p-value function

For $\theta_0=-4,\ 0,\ 2,\ 4,\ 5,\ 6$ points the p-values are 0.004, 0.52, 0.64, 0.12, 0.033, and 0.007. The estimate is 1.16 points, and the curve is symmetric around it, so the p-value of 0 equals that of 2.32 points: both are 1.16 points from the estimate and the standard error is the same in both directions. The 90% interval is $-1.80$ to 4.12 points.

In [21]:
# 1. P-value function and the 90% interval
for theta0 in (-4, 0, 2, 4, 5, 6):
    print("Hypothesized", theta0, "points: p =", round(2 * norm.sf(abs((theta - theta0 / 100) / se)), 4))
print("Same p-value as 0:", 0, "and", round(200 * theta, 2), "; 90% CI:", round(100 * (theta - 1.645 * se), 2), "to", round(100 * (theta + 1.645 * se), 2))

# 2. The two hypothesized differences whose p-value is exactly 0.52: the estimate plus and minus se times the normal quantile
z_52 = norm.isf(0.52 / 2)
print("Differences with p = 0.52:", round(100 * (theta - z_52 * se), 2), "and", round(100 * (theta + z_52 * se), 2), "points")

Hypothesized -4 points: p = 0.0042
Hypothesized 0 points: p = 0.5193
Hypothesized 2 points: p = 0.6415
Hypothesized 4 points: p = 0.1151
Hypothesized 5 points: p = 0.0331
Hypothesized 6 points: p = 0.0072
Same p-value as 0: 0 and 2.32 ; 90% CI: -1.8 to 4.12
Differences with p = 0.52: 0.0 and 2.32 points


### 15.18 Exercise 6. S-values

The S-values are 0.94 bits for $p=0.52$ (about one head in a row), 4.32 bits for $p=0.05$ (about four heads), 4.92 bits for $p=0.033$ (about five), and 9.97 bits for $p=0.001$ (about ten). A p-value of 0.52 is hardly more surprising than a single coin toss; $p=0.001$ is as surprising as ten heads in a row.

In [22]:
# 1. S-values and their meaning in coin tosses
for p in (0.52, 0.05, 0.033, 0.001):
    print("p =", p, ": S-value", round(-log2(p), 2), "bits; heads in a row: about", round(-log2(p)))

p = 0.52 : S-value 0.94 bits; heads in a row: about 1
p = 0.05 : S-value 4.32 bits; heads in a row: about 4
p = 0.033 : S-value 4.92 bits; heads in a row: about 5
p = 0.001 : S-value 9.97 bits; heads in a row: about 10


### 15.18 Exercise 7. Significant versus important

With the observed difference and failure rates fixed, the p-value of the test of no difference is 0.52 with 591 patients per arm, 0.15 with 3,000, 0.0085 with 10,000, and 0.00001 with 30,000. The estimate, 1.2 points, is the same in all four; only its precision changed. A tiny p-value can sit on a difference that is small compared with the 5-point margin, so significance is not importance.

In [23]:
# 1. The p-value of no difference for larger trials with the observed difference fixed
for n_arm in (591, 3000, 10000, 30000):
    z_n = theta / sqrt((p_s * (1 - p_s) + p_c * (1 - p_c)) / n_arm)
    print("Patients per arm", n_arm, ": two-sided p =", "%.2g" % (2 * norm.sf(abs(z_n))))

Patients per arm 591 : two-sided p = 0.52
Patients per arm 3000 : two-sided p = 0.15
Patients per arm 10000 : two-sided p = 0.0085
Patients per arm 30000 : two-sided p = 5.1e-06


### 15.18 Exercise 8. False significant results

With 20% of the hypotheses real, the share of false significant results is 0.20 for power 0.80, 0.29 for power 0.50, and 0.50 for power 0.20. For power 0.80, the share of real hypotheses at which the false share is 5% is 54.3%: only when more than half of the hypotheses tested are real does a significant result at the 5% level have a 5% chance of being false.

In [24]:
# 1. Share of false significant results with 20% real hypotheses, and the share of real hypotheses that gives 5%
for power in (0.8, 0.5, 0.2):
    print("Power", power, ": share false", round(0.8 * 0.05 / (0.8 * 0.05 + 0.2 * power), 3))
share_real = 0.95 / 1.75                                           # solves 0.95 * (1 - r) = 0.80 * r
print("Real hypotheses giving 5% false (power 0.8):", round(100 * share_real, 1), "%")

Power 0.8 : share false 0.2
Power 0.5 : share false 0.286
Power 0.2 : share false 0.5
Real hypotheses giving 5% false (power 0.8): 54.3 %


### 15.18 Exercise 9. The twelve sites

The differences (short minus standard) range from $-10.6$ points at S09 to $+11.8$ at S01. One site has $p<0.05$: S09, with $p=0.020$. The Bonferroni threshold is $0.05/12=0.0042$, and no site passes it. The chance of at least one $p<0.05$ among 12 independent tests of true null hypotheses is $1-0.95^{12}=0.46$.

In [25]:
# 1. Differences and p-values by site, and the Bonferroni threshold
below_005 = []
below_bonferroni = []
for site, group in df.groupby("site"):
    a = group[group["allocated"] == "short"]["fail"]
    b = group[group["allocated"] == "standard"]["fail"]
    table = [[int(a.sum()), len(a) - int(a.sum())], [int(b.sum()), len(b) - int(b.sum())]]
    p_site = stats.fisher_exact(table)[1]
    if p_site < 0.05:
        below_005.append(site)
    if p_site < 0.05 / 12:
        below_bonferroni.append(site)
    print(site, ": difference", round(100 * (a.mean() - b.mean()), 1), "points; Fisher p =", round(p_site, 4))
print("Sites with p < 0.05:", below_005, "; sites below the Bonferroni threshold:", below_bonferroni)
print("Bonferroni threshold:", round(0.05 / 12, 4), "; chance of at least one p < 0.05 in 12 independent tests:", round(1 - 0.95 ** 12, 3))

S01 : difference 11.8 points; Fisher p = 0.1058
S02 : difference 7.2 points; Fisher p = 0.3222
S03 : difference 3.4 points; Fisher p = 0.751
S04 : difference 3.4 points; Fisher p = 0.7577
S05 : difference 1.0 points; Fisher p = 1.0
S06 : difference -1.6 points; Fisher p = 1.0
S07 : difference 1.1 points; Fisher p = 1.0
S08 : difference -4.2 points; Fisher p = 0.5013
S09 : difference -10.6 points; Fisher p = 0.0204
S10 : difference 7.2 points; Fisher p = 0.4038
S11 : difference 0.8 points; Fisher p = 1.0
S12 : difference -5.2 points; Fisher p = 0.5092
Sites with p < 0.05: ['S09'] ; sites below the Bonferroni threshold: []
Bonferroni threshold: 0.0042 ; chance of at least one p < 0.05 in 12 independent tests: 0.46


### 15.18 Exercise 10. Four looks

In 4,000 shuffles with four looks, the rule that stops at the first look with $|z|>1.96$ rejected the true null hypothesis in 12.9%; with the Pocock boundary 2.361 the rate was 5.0%. With four looks the inflation is smaller than with six looks (16.0%), and the Pocock boundary restores the 5%.

In [26]:
# 1. Four looks (every 300 patients): rejection rates in 4,000 shuffles
rng = np.random.default_rng(15)
labels = (df["allocated"] == "short").values
y = df["fail"].values
hit_196 = 0
hit_pocock = 0
for repetition in range(4000):
    shuffled = rng.permutation(labels)
    z_four = []
    for n_look in (300, 600, 900, 1200):
        is_short = shuffled[:n_look]
        yy = y[:n_look]
        p_all = yy.mean()                                          # failure rate of all patients so far (standard error under the null hypothesis)
        z_four.append(abs(yy[is_short].mean() - yy[~is_short].mean()) / sqrt(p_all * (1 - p_all) * (1 / is_short.sum() + 1 / (n_look - is_short.sum()))))
    hit_196 = hit_196 + (max(z_four) > 1.96)
    hit_pocock = hit_pocock + (max(z_four) > 2.361)
print("Four looks at 1.96:", round(100 * hit_196 / 4000, 1), "%; at 2.361:", round(100 * hit_pocock / 4000, 1), "%")

Four looks at 1.96: 12.9 %; at 2.361: 5.0 %


### 15.18 Exercise 11. Boundaries for four looks

For four equally spaced looks, the Pocock boundary is 2.361, and the O'Brien-Fleming boundaries are 4.049, 2.863, 2.337, and 2.024. The Lan-DeMets spending function $\alpha(t)=2-2\Phi(1.96/\sqrt t)$ spends 0.0001 by $t=0.25$, 0.0056 by 0.5, 0.0236 by 0.75, and 0.05 at the end.

In [27]:
# 1. Boundaries for four equally spaced looks and the spending function
# The correlation of the statistics at looks i and j is sqrt(t_i / t_j); brentq finds the boundary that makes the probability of crossing at any look 0.025
t4 = np.arange(1, 5) / 4
mv4 = multivariate_normal(mean=np.zeros(4), cov=np.sqrt(np.minimum.outer(t4, t4) / np.maximum.outer(t4, t4)), seed=1, abseps=1e-8, releps=1e-8)
pocock4 = brentq(lambda c: 1 - mv4.cdf(np.array([c] * 4)) - 0.025, 1.5, 4)
obf4 = brentq(lambda c: 1 - mv4.cdf(np.array([c * sqrt(4 / k) for k in (1, 2, 3, 4)])) - 0.025, 1.5, 4)
print("Pocock:", round(pocock4, 3), "; O'Brien-Fleming:", [round(obf4 * sqrt(4 / k), 3) for k in (1, 2, 3, 4)])
print("Spending at 0.25, 0.5, 0.75, 1:", [round(float(2 - 2 * norm.cdf(1.96 / sqrt(t_k))), 4) for t_k in (0.25, 0.5, 0.75, 1)])

Pocock: 2.361 ; O'Brien-Fleming: [4.049, 2.863, 2.337, 2.024]
Spending at 0.25, 0.5, 0.75, 1: [0.0001, 0.0056, 0.0236, 0.05]


### 15.18 Exercise 12. An early stop and conditional power

At the second look the boundary is 2.454, and $Z=2.6$ crosses it. The fixed-sample two-sided $p=0.0093$ treats the result as a single test, and ignores that the trial could have stopped at any of three looks and was allowed 5% in total; the p-value that respects the design is larger. The estimate of a trial that stops at a high early boundary is likely to be too far from the null, in the direction that favors the new treatment. With evidence 1.5 at $t=2/3$ and a final boundary 2.004, the conditional power is $\Phi\bigl((1.5/0.816-2.004)/0.577\bigr)=0.39$.

In [28]:
# 1. Fixed-sample p-value of Z = 2.6, and the p-value that respects the design (first look, then the second look at least as extreme)
print("Fixed-sample two-sided p of Z = 2.6:", round(2 * norm.sf(2.6), 4), "; boundary at the second look:", round(obf_c * sqrt(3 / 2), 3))
c_first = obf_c * sqrt(3)                                          # boundary at the first look
r = sqrt(1 / 2)                                                    # correlation between the first and the second look
def_cdf = lambda a: quad(lambda x: norm.pdf(x) * norm.cdf((2.6 - r * x) / sqrt(1 - r ** 2)), -np.inf, a)[0]     # P(Z1 < a and Z2 < 2.6)
inside_first = (norm.cdf(c_first) - def_cdf(c_first)) - (norm.cdf(-c_first) - def_cdf(-c_first))               # P(-c < Z1 < c and Z2 > 2.6)
print("P-value of the design:", round(2 * norm.sf(c_first) + 2 * inside_first, 4))

# 2. Conditional power with evidence 1.5 at t = 2/3 and final boundary 2.004
print("Conditional power:", round(norm.cdf((1.5 / sqrt(2 / 3) - 2.004) / sqrt(1 / 3)), 3))

Fixed-sample two-sided p of Z = 2.6: 0.0093 ; boundary at the second look: 2.454
P-value of the design: 0.0096
Conditional power: 0.386


---
*Synthetic data (seed 1964, `generate_data.py`). Teaching values only; nothing here is an estimate for a real trial.*